# 🗺️ Game Terrain Generator — PromptFlow Engine

### Architecture (Direct In-Process Mode)
```
┌──────────────────────────────────────────────────┐
│  Everything runs in ONE Python process.           │
│  No HTTP server. No sockets. No timeouts.         │
│                                                   │
│  Phase 1 (LLM thinking)                           │
│  Llama-3.2-3B-Instruct-uncensored  ~2GB GPU       │
│    └─ sd.generate_image() called directly          │
│                                                   │
│  ⬇  offload LLM to CPU (frees ~2GB VRAM)          │
│                                                   │
│  Phase 2 (Image gen)                              │
│  Z-Image Turbo GGUF  ~10GB GPU                    │
│    └─ sd.generate_image() called directly          │
│    └─ gc.collect() + cuda.empty_cache() each image │
│    └─ Skip already-generated (resume support)      │
│                                                   │
│  🔄 Background keep-alive runs automatically       │
└──────────────────────────────────────────────────┘
```

**Why direct mode?** The HTTP server architecture (model_server.py)
caused connection drops after ~5 images. By loading both models directly
in the notebook process and passing them to the interpreter, we eliminate
all network-related failure modes: no sockets, no serialization, no
timeouts, no base64 round-trips for images.

**Resume support:** If the run fails, re-run Cell 5 — already-generated images are skipped.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Install dependencies
# ═══════════════════════════════════════════════════════════════
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 -q
!pip install stable-diffusion-cpp-python huggingface_hub rich pyyaml requests pillow openai -q
print('✅ Dependencies installed')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Download all model files from HuggingFace
# ═══════════════════════════════════════════════════════════════
import os
from huggingface_hub import hf_hub_download

# ── Uncensored local LLM (creative text generation) ──────────
LOCAL_LLM_REPO = 'bartowski/Llama-3.2-3B-Instruct-uncensored-GGUF'
LOCAL_LLM_FILE = 'Llama-3.2-3B-Instruct-uncensored-Q4_K_M.gguf'

# ── Z-Image Turbo components ─────────────────────────────────
DIFF_REPO     = 'unsloth/Z-Image-Turbo-GGUF'
DIFF_FILE     = 'z-image-turbo-Q4_K_M.gguf'
LLM_DIFF_REPO = 'unsloth/Qwen3-4B-Instruct-2507-GGUF'
LLM_DIFF_FILE = 'Qwen3-4B-Instruct-2507-Q4_K_M.gguf'
VAE_REPO      = 'black-forest-labs/FLUX.1-schnell'
VAE_FILE      = 'ae.safetensors'

print('⏳ Downloading models (this may take a few minutes)...')
local_llm_path = hf_hub_download(repo_id=LOCAL_LLM_REPO, filename=LOCAL_LLM_FILE)
diff_path      = hf_hub_download(repo_id=DIFF_REPO,      filename=DIFF_FILE)
llm_diff_path  = hf_hub_download(repo_id=LLM_DIFF_REPO,  filename=LLM_DIFF_FILE)
vae_path       = hf_hub_download(repo_id=VAE_REPO,       filename=VAE_FILE)

print(f'\n✅ Local uncensored LLM : {local_llm_path}')
print(f'✅ Z-Image diffusion    : {diff_path}')
print(f'✅ Z-Image LLM component: {llm_diff_path}')
print(f'✅ VAE                  : {vae_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Load models DIRECTLY in this process
#   No HTTP server. Both models live in this Python process.
#   The interpreter calls them directly — zero network overhead.
# ═══════════════════════════════════════════════════════════════
import gc

# ── Load local uncensored LLM ─────────────────────────────────
print('⏳ Loading local LLM...')
from llama_cpp import Llama

llm = Llama(
    model_path=local_llm_path,
    n_gpu_layers=99,
    n_ctx=4096,
    verbose=False,
    chat_format='llama-3',
)
print('✅ Local LLM loaded on GPU')

# Quick test
test = llm.create_chat_completion(
    messages=[{'role': 'user', 'content': 'Name 3 terrain types. One line each.'}],
    temperature=0.9, max_tokens=100,
)
print(f'   Test: {test["choices"][0]["message"]["content"][:80]}...')

# ── Load Z-Image Turbo ────────────────────────────────────────
print('\n⏳ Loading Z-Image Turbo...')
from stable_diffusion_cpp import StableDiffusion

sd = StableDiffusion(
    diffusion_model_path=diff_path,
    llm_path=llm_diff_path,
    vae_path=vae_path,
    offload_params_to_cpu=True,
    diffusion_flash_attn=True,
)
print('✅ Z-Image Turbo loaded')

# GPU status
try:
    import torch
    alloc = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'\n📊 GPU: {alloc:.1f}/{total:.1f} GB used')
except Exception:
    pass

print('\n✅ Both models loaded in-process. No HTTP server needed.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Quick smoke test — generate one image directly
# ═══════════════════════════════════════════════════════════════
from IPython.display import display, Image as IPyImage
import gc, io

print('⏳ Generating test image...')
gc.collect()

test_images = sd.generate_image(
    prompt='a grassy terrain tile for an RTS game, isometric view, black background',
    negative_prompt='people, text, watermark',
    width=1024, height=1024,
    sample_steps=4, cfg_scale=1.0,
    sample_method='euler', seed=42,
)

# Save and display
test_images[0].save('test_terrain.png')
display(IPyImage(filename='test_terrain.png', width=512))
print('✅ Direct image generation works. No HTTP needed.')

# Cleanup
for img in test_images:
    img.close()
del test_images
gc.collect()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: 🚀 Run the PromptFlow (DIRECT MODE)
#
# Both `sd` and `llm` objects are passed directly to the
# interpreter. All calls happen in-process:
#   - llm.create_chat_completion() for text generation
#   - sd.generate_image() for image generation
#   - gc.collect() + cuda.empty_cache() between every image
#
# No HTTP. No sockets. No timeouts. No base64 round-trips.
#
# ♻️ RESUME: re-run this cell to skip already-generated images.
# 🔄 KEEP-ALIVE: background thread starts automatically.
# ═══════════════════════════════════════════════════════════════
import sys, os
sys.path.insert(0, '.')
from interpreter.interpreter import PromptFlowInterpreter

# ── CONFIGURE YOUR GAME ───────────────────────────────────────
GAME_CONTEXT = """
A photorealistic RTS set during the late Cretaceous. Players control dinosaur factions
across lush jungles, volcanic plains, coastal marshlands, and highland cliffs.
Art style: photorealistic CGI, cinematic grade. Each terrain tile is a floating
isometric platform with the dirt cross-section visible on all sides. Black background.
"""

TERRAIN_INSTRUCTIONS = """
Include: flat grass, jungle floor with roots, volcanic rock, coastal marsh, cliff edges
(all 4 sides + 4 corners), river crossing, volcanic crater. Edge-connectable tiles.
Consistent lighting: overcast midday from upper-left.
"""

TERRAIN_ASSETS_TO_MAKE = 25
OUTPUT_DIR = 'output/terrain_tiles'
SAMPLE_IMAGE = 'sample_terrain.png'    # optional: reference image for img2img
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')  # only if provider: openai
# ──────────────────────────────────────────────────────────────

pf = PromptFlowInterpreter(
    promptflow_path='promptflows/game_terrain_generator.promptflow',
    cli_inputs={
        'game_context':               GAME_CONTEXT,
        'terrain_asset_instructions': TERRAIN_INSTRUCTIONS,
        'terrain_assets_to_make':     TERRAIN_ASSETS_TO_MAKE,
        'out_dir':                    OUTPUT_DIR,
    },
    openai_api_key=OPENAI_API_KEY,
    sample_image_path=SAMPLE_IMAGE if os.path.exists(SAMPLE_IMAGE) else None,
    dry_run=False,
    # ── DIRECT MODE: pass model objects, bypass HTTP entirely ──
    sd_instance=sd,
    llm_instance=llm,
)

results = pf.run()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: Gallery view — browse all generated tiles
# ═══════════════════════════════════════════════════════════════
from pathlib import Path
from IPython.display import display, HTML

images = sorted(Path(OUTPUT_DIR).glob('terrain_*.png'))
print(f'🖼️  {len(images)} terrain tiles in {OUTPUT_DIR}')

html = "<div style='display:flex;flex-wrap:wrap;gap:10px;padding:8px;background:#111'>"
for p in images:
    html += f"<div style='text-align:center'><img src='{p}' width='200' style='border:1px solid #444;border-radius:3px'><br><span style='font-size:10px;color:#888;font-family:monospace'>{p.stem}</span></div>"
html += '</div>'
display(HTML(html))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: GPU memory status
# ═══════════════════════════════════════════════════════════════
import gc
gc.collect()
try:
    import torch
    torch.cuda.empty_cache()
    alloc = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_mem / 1e9
    free  = total - alloc
    print(f'GPU: {alloc:.1f}/{total:.1f} GB used  ({free:.1f} GB free)')
except Exception as e:
    print(f'GPU info unavailable: {e}')

try:
    import subprocess
    nv = subprocess.run(
        ['nvidia-smi', '--query-gpu=utilization.gpu,temperature.gpu,power.draw',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True, timeout=3)
    if nv.returncode == 0:
        parts = [x.strip() for x in nv.stdout.strip().split(',')]
        print(f'GPU util: {parts[0]}%  temp: {parts[1]}°C  power: {parts[2]}W')
except Exception:
    pass